# TrustLens multi-source tampering detector retraining

This notebook streams the public combined tampering dataset plus fully synthetic hard negatives from Tiny-GenImage and writes only the trained checkpoint to Drive. It requires the repository training script and compact-pack provenance so every evaluation source group is excluded from training. The candidate never overwrites the deployed model automatically.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib

drive.mount('/content/drive')
print('Select train_tampering_detector_streaming.py')
script_upload = files.upload()
assert 'train_tampering_detector_streaming.py' in script_upload

print('Select compact_provenance.csv')
provenance_upload = files.upload()
assert 'compact_provenance.csv' in provenance_upload

BACKEND_DIR = Path('/content/backend')
SCRIPT = BACKEND_DIR / 'scripts' / 'train_tampering_detector_streaming.py'
PROVENANCE = BACKEND_DIR / 'calibration' / 'compact_provenance.csv'
SCRIPT.parent.mkdir(parents=True, exist_ok=True)
PROVENANCE.parent.mkdir(parents=True, exist_ok=True)
SCRIPT.write_bytes(script_upload[SCRIPT.name])
PROVENANCE.write_bytes(provenance_upload[PROVENANCE.name])
digest = hashlib.sha256(SCRIPT.read_bytes()).hexdigest()
assert digest == '51e3efe5d7bc18435d5f235be1963340c114648e3594351c236badbbe5902528', digest
print('Correct training script verified:', digest)
print('Evaluation provenance:', PROVENANCE)

In [ ]:
%pip install -q "datasets>=4,<5" "scikit-learn>=1.5,<2" "tqdm>=4.66"

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess
import sys

OUTPUT_DIR = Path('/content/drive/MyDrive/TrustLens/trustlens_MJ-1/backend/training_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = OUTPUT_DIR / 'tampering_multisource_hardnegative_candidate.pth'
REPORT = OUTPUT_DIR / 'tampering_multisource_hardnegative_report.json'

command = [
    sys.executable, str(SCRIPT),
    '--output', str(CHECKPOINT),
    '--report', str(REPORT),
    '--exclude-provenance', str(PROVENANCE),
    '--train-per-class', '2000',
    '--validation-per-class', '400',
    '--synthetic-negative-fraction', '0.5',
    '--epochs', '6',
    '--batch-size', '32',
]
subprocess.run(command, cwd=BACKEND_DIR, check=True)
print('Candidate:', CHECKPOINT)
print('Report:', REPORT)

In [ ]:
import json

report = json.loads(REPORT.read_text())
size_mb = CHECKPOINT.stat().st_size / 1024 / 1024
print('Checkpoint size:', round(size_mb, 2), 'MB')
print('Best validation AUROC:', round(report['best_validation_auroc'], 6))
print('Excluded evaluation groups:', report['partition']['excluded_evaluation_groups'])
print('Synthetic negative fraction:', report['partition']['synthetic_negative_fraction'])
print('Promotion status:', report['promotion_status'])
assert size_mb > 35
assert report['partition']['excluded_evaluation_groups'] >= 90
assert report['partition']['synthetic_negative_fraction'] == 0.5
assert report['promotion_status'] == 'CANDIDATE_REQUIRES_EXTERNAL_CALIBRATION'

## After training

Download `tampering_multisource_hardnegative_candidate.pth` and `tampering_multisource_hardnegative_report.json` from the Drive `training_outputs` directory. Do not rename or deploy the checkpoint before the untouched TrustLens external gates pass.